In [18]:
import requests

##### robots.txt-checking if we are even allowed to scrape

In [19]:
url = 'http://books.toscrape.com/robots.txt'

In [20]:
print(requests.get(url).text)

<html>
<head><title>404 Not Found</title></head>
<body>
<center><h1>404 Not Found</h1></center>
<hr><center>nginx/1.21.6</center>
</body>
</html>



In [21]:
url = 'https://www.ifixit.com/robots.txt'

In [22]:
print(requests.get(url).text)

# As a condition of accessing this website, you agree to abide by the
# following content signals:
#
# (a) If a content-signal = yes, you may collect content for the corresponding use.
# (b) If a content-signal = no, you may not collect content for the corresponding use.
# (c) If the website operator does not include a content signal for a
#     corresponding use, the website operator neither grants nor restricts
#     permission via content signal with respect to the corresponding use.
#
# The content signals and their meanings are:
#
# search: building a search index and providing search results (e.g., returning
#   hyperlinks and short excerpts from your website's contents). Search does not
#   include providing AI-generated search summaries.
# ai-input: inputting content into one or more AI models (e.g., retrieval
#   augmented generation, grounding, or other real-time taking of content for
#   generative AI search answers).
# ai-train: training or fine-tuning AI models.
#
# ANY RE

In [23]:
from urllib.robotparser import RobotFileParser

In [24]:
rp = RobotFileParser()
rp.set_url('http://books.toscrape.com/robots.txt')
rp.read()

print(rp.can_fetch('*', 'http://books.toscrape.com/catalogue/page-2.html'))

True


In [ ]:
rp = RobotFileParser()
rp.set_url(url)
rp.read()

print(rp.can_fetch('*', 'https://www.ifixit.com/Device/Acer_Aspire_1_A114-31'))
rp.can_fetch('*', 'https://www.ifixit.com/Search?query=battery')

<div style='color: green'>urllib, in our setup, instead asked Windows itself: "give me your list of trusted certificates." Windows tried to hand over its certificate store, and one of the certificates in that store was malformed/unreadable — so the whole loading process crashed before it even got to actually contacting iFixit. </div>
<div style='color: red'> SSLError !!!</div>                             

In [ ]:
url = 'https://www.ifixit.com/robots.txt'

response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0 Chrome/124.0.0.0'})
rp = RobotFileParser()
rp.parse(response.text.splitlines()) 

print(rp.can_fetch('*', 'https://www.ifixit.com/Device/Acer_Aspire_1_A114-31'))
print(rp.can_fetch('*', 'https://www.ifixit.com/Search?query=battery'))

True
False


##### Timeouts and connection errors

In [ ]:
try:
    response = requests.get('https://httpbin.org/delay/5', timeout=10)
    print(response.status_code)
except requests.exceptions.Timeout:
    print('Request timed out!')

200


In [ ]:
try:
    response = requests.get('https://httpbin.org/delay/5', timeout=2)
    print(response.status_code)
except requests.exceptions.Timeout:
    print('Request timed out!')

Request timed out!


##### Handling bad status codes

In [ ]:
try:
    response = requests.get('https://httpbin.org/status/404', timeout=5)
    response.raise_for_status()
    print('Success:', response.status_code)
except requests.exceptions.HTTPError as e:
    print('HTTP error occured:', e)

HTTP error occured: 503 Server Error: Service Temporarily Unavailable for url: https://httpbin.org/status/404


In [ ]:
import requests

response = requests.get('https://httpbin.org/status/404', timeout=5)
print("Status code:", response.status_code)
print("Response headers:", dict(response.headers))
print("Response body:", response.text[:300])

Status code: 503
Response headers: {'Server': 'awselb/2.0', 'Date': 'Sat, 08 Aug 2026 05:38:20 GMT', 'Content-Type': 'text/html', 'Content-Length': '162', 'Connection': 'keep-alive'}
Response body: <html>
<head><title>503 Service Temporarily Unavailable</title></head>
<body>
<center><h1>503 Service Temporarily Unavailable</h1></center>
</body>
</html>



In [ ]:
import time

def get_with_retry(url, max_retries=3, backoff=2):
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(url, timeout=5)
            response.raise_for_status()
            return response
        except requests.exceptions.RequestException as e:
            print(f'Attempt {attempt} failed: {e}')
            if attempt < max_retries:
                wait = backoff ** attempt
                print(f'Retrying in {wait} seconds')
                time.sleep(wait)
    print('All retires failed')
    return None

In [ ]:
response = get_with_retry('http://httpbin.org/status/404')

if response:
    print(response.status_code)

Attempt 1 failed: 503 Server Error: Service Temporarily Unavailable for url: http://httpbin.org/status/404
Retrying in 2 seconds
Attempt 2 failed: 503 Server Error: Service Temporarily Unavailable for url: http://httpbin.org/status/404
Retrying in 4 seconds
Attempt 3 failed: 503 Server Error: Service Temporarily Unavailable for url: http://httpbin.org/status/404
All retires failed


In [ ]:
import requests

session = requests.Session()
session.headers.update({'User-Agent': 'Mozilla/5.0 Chrome/124.0.0.0'})

response1 = session.get('http://books.toscrape.com/')
response2 = session.get('http://books.toscrape.com/catalogue/page-2.html')

In [28]:
import random
import time

time.sleep(random.uniform(1, 3))

### Let's put everything together

In [29]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser
import time
import random
import csv

In [30]:
BASE_URL = 'http://books.toscrape.com'
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/124.0.0.0'}
OUTPUT_FILE = 'books.csv'

In [31]:
robots_response = requests.get(urljoin(BASE_URL, '/robots.txt'), headers=HEADERS)
rp = RobotFileParser()

if robots_response.status_code == 200:
    rp.parse(robots_response.text.splitlines())
else:
    rp = None

In [32]:
def allowed(url):
    if rp is None:
        return True
    return rp.can_fetch('*', url)

In [33]:
session = requests.Session()
session.headers.update(HEADERS)